In [ ]:
# 현재 노트북이 사용하는 파이썬 환경 확인
import sys

print("파이썬 경로:", sys.executable)
print("파이썬 버전:", sys.version)

In [ ]:
# 현재 노트북의 파이썬 환경에 필요한 패키지 설치
%pip install langchain langchain-openai python-dotenv

In [ ]:
# 설치한 패키지를 정상적으로 불러올 수 있는지 확인
from importlib.metadata import version
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv

print("langchain:", version("langchain"))
print("langchain-openai:", version("langchain-openai"))
print("랭체인 준비 완료!")

In [ ]:
from pathlib import Path
from dotenv import dotenv_values
from langchain_openai import ChatOpenAI

# 노트북 실행 위치와 상위 폴더에서 기존 .env 찾기
current_dir = Path.cwd()
api_key = None
for folder in [current_dir, *current_dir.parents]:
    candidates =  [
        folder / ".env",
        folder / "chap08" / ".env",
        folder / "chap07" / ".env",
    ]

    for env_path in candidates:
        if env_path.is_file():
            settings = dotenv_values(env_path)
            key = settings.get("OPENAI_API_KEY")

            if key and key.strip():
                api_key = key.strip()
                print("API 키를 불러온 파일:", env_path)
                break
    
    if api_key:
        break

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY를 찾지 못했습니다. 기존 .env 위치를 확인해주세요."
    )

# 기존 실습에서 사용한 모델 연결
model = ChatOpenAI(
    model="gpt-4o",
    api_key=api_key,
    temperature=0.7,
    timeout=30,
    max_retries=1,
)

print("랭체인 모델 준비 완료!")

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from IPython.display import display, Markdown

messages = [
    SystemMessage(
        content=(
            "너는 친절한 AI 비서야. 한국어로 짧고 쉽게 설명해줘."
            "각 문장을 별도 문단으로 작성하고, 문장 사이에 빈 줄을 넣어줘."
        )

    ),
    HumanMessage(
        content="랭체인이 무엇인지 로봇을 만드는 대학생에게 세 문장으로 설명해줘."
    ),
]

# 실제 API 요청: 실행할 때 사용료가 발생함
response = model.invoke(messages)

# 응답에서 답변 내용만 출력
display(Markdown(response.content))

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from IPython.display import display, Markdown

# 새로운 대화 기록 만들기
messages = [
    SystemMessage(
        content=(
            "너는 친절한 AI 비서야. 한국어 짧고 자연스럽게 답해줘."
            "각 문장을 별도 문단으로 작성하고, 문장 사이에 빈 줄을 넣어줘."
        )
    )
]

# 사용자 메시지를 기록에 추가
messages.append(
    HumanMessage(content="내 이름은 김성환이고, 정보통신공학, 메카공학을 공부했어.")
)

# 전체 대화 기록을 전달해서 답변 받기
response = model.invoke(messages)

# AI 답변도 기록에 추가
messages.append(response)

In [ ]:
# 기존 기록에 후속 질문 추가
messages.append(
    HumanMessage(content="내 이름과 내가 공부한 전공이 뭐였지?")
)

# 이전 대화까지 함께 전달
response = model.invoke(messages)

# 새 답변도 저장
messages.append(response)

display(Markdown(response.content))

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import SystemMessage, HumanMessage
from IPython.display import display, Markdown

# 대화방별 기록을 보관하는 공간
store = {}

def get_session_history(session_id: str):
    """"대화방이 없으면 새로 만들고, 해당 대화방의 기록을 반환한다."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory(
            messages=[
                SystemMessage(
                    content=(
                        "한국어로 짧고 친절하게 답해줘. "
                        "대화에 없는 사용자 정보는 추측하지 마."
                    )
                )
            ]
        )
    return store[session_id]

# 모델에 대화 기록 관리 기능 연결
chat_with_history = RunnableWithMessageHistory(
    model,
    get_session_history,
)

# 사용할 대화방 이름
config = {"configurable": {"session_id": "YOUR_USERhwan_01"}}

print("대화 기록 관리 준비 완료!")

In [ ]:
response = chat_with_history.invoke(
    [
        HumanMessage(
            content="내 이름은 김성환이고, 지금 ESP32-CAM 프로젝트를 하고 있어."
        )
    ],
    config=config,
)

display(Markdown(response.content))

In [ ]:
response =chat_with_history.invoke(
    [HumanMessage(content="내 이름과 지금 진행하는 프로젝트가 뭐였지?")],
    config=config,
)

display(Markdown(response.content))

In [ ]:
new_config = {"configurable": {"session_id": "new_room"}}

response = chat_with_history.invoke(
    [HumanMessage(content="내 이름과 지금 진행하는 프로젝트가 뭐였지?")],
    config=new_config,
)

display(Markdown(response.content))

In [ ]:
from IPython.display import display, Markdown

# 답변을 표시할 공간 만들기
answer_display = display(
    Markdown("답변을 준비하고 있어요..."),
    display_id=True,
)

answer = ""

# 원래 대화방의 기록을 이용해서 스트림으로 답변받기
for chunk in chat_with_history.stream(
    [
        HumanMessage(
            content="내 프로젝트를 소개하는 문장을 세 문장으로 작성해줘."
                    "아직 말하지 않은 세부 기능은 지어내지마."
                    "각 문장을 별도 문단으로 작성해"
                    
        )
    ],
    config=config,
):
    # 받은 텍스트 조각을 기존 답변에 이어 붙이기
    if isinstance(chunk.content, str) and chunk.content:
        answer += chunk.content

        # 같은 출력 공간을 갱신하기
        answer_display.update(Markdown(answer))